# AVERAGE THE DATA IN THE SAME COORDINATES

In [1]:
import sys
print(sys.executable)

/usr/bin/python


In [2]:
#we import all they key libraries needed in this notebook
import numpy as np
import pandas as pd
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import matplotlib as matplotlib

### Required files:
We load the data and save it in a pandas dataframe.
- ***./csv/features/bact_data.csv***

In [3]:
nifh_df = pd.read_csv("./csv/features/bact_data.csv") 
print(nifh_df.columns)
nifh_df.info()

Index(['Unnamed: 0', 'Trichodesmium nifH Gene (x106 copies m-3)',
       'UCYN-A nifH Gene (x106 copies m-3)',
       'UCYN-B nifH Gene (x106 copies m-3)',
       'Heterocyst (Richelia & Calotrhix) Biomass (mg C m-3)',
       'Gamma A nifH Gene (x106 copies/m3)', 'LATITUDE', 'LONGITUDE'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2926 entries, 0 to 2925
Data columns (total 8 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   Unnamed: 0                                            2926 non-null   int64  
 1   Trichodesmium nifH Gene (x106 copies m-3)             2127 non-null   float64
 2   UCYN-A nifH Gene (x106 copies m-3)                    2258 non-null   float64
 3   UCYN-B nifH Gene (x106 copies m-3)                    1760 non-null   float64
 4   Heterocyst (Richelia & Calotrhix) Biomass (mg C m-3)  299 non-null    float64
 

### Not all columns have enough data
Obviously we can tell that some columns like Gamma-A have not enough data, so we would rather not use it for our model. 
### There are duplicate coordinates
Also, for each of these types of bacteria there might be duplicate coordinates. In such cases we want to average the data on that coordinate. I have a suspicion that if we just group the main dataframe by coordinates and take mean it can lead to a lot of null values. I will test this theory.

**2 methods for averaging**:
1. simple average after group by
2. separate all columns average and join back together

In [4]:
#this takes an average in a simple way
nifh_simple_avg = nifh_df.groupby(by=['LATITUDE', 'LONGITUDE']).mean()
print(nifh_simple_avg.columns)
nifh_simple_avg.info()

Index(['Unnamed: 0', 'Trichodesmium nifH Gene (x106 copies m-3)',
       'UCYN-A nifH Gene (x106 copies m-3)',
       'UCYN-B nifH Gene (x106 copies m-3)',
       'Heterocyst (Richelia & Calotrhix) Biomass (mg C m-3)',
       'Gamma A nifH Gene (x106 copies/m3)'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
MultiIndex: 1025 entries, (-76.0, 34.0) to (120.0, 25.0)
Data columns (total 6 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   Unnamed: 0                                            1025 non-null   float64
 1   Trichodesmium nifH Gene (x106 copies m-3)             811 non-null    float64
 2   UCYN-A nifH Gene (x106 copies m-3)                    844 non-null    float64
 3   UCYN-B nifH Gene (x106 copies m-3)                    498 non-null    float64
 4   Heterocyst (Richelia & Calotrhix) Biomass (mg C m-3)  117 non-null    float64
 5   

In [5]:
nifh_simple_avg.describe()

,Unnamed: 0,Trichodesmium nifH Gene (x106 copies m-3),UCYN-A nifH Gene (x106 copies m-3),UCYN-B nifH Gene (x106 copies m-3),Heterocyst (Richelia & Calotrhix) Biomass (mg C m-3),Gamma A nifH Gene (x106 copies/m3)
count,1025.000000,8.110000e+02,8.440000e+02,498.000000,117.000000,69.000000
mean,2427.080182,6.337915e+04,3.619769e+03,266.111383,4.471009,7.748551
std,1286.764146,1.373748e+06,8.071648e+04,3946.598824,16.774477,11.478492
min,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000
25%,1795.888889,2.306250e+00,1.937500e-01,0.000000,0.000800,0.350000
50%,2542.000000,3.981100e+01,5.130000e+00,0.444250,0.007450,1.120000
75%,2923.000000,2.537567e+02,8.080375e+01,8.817875,0.057300,10.510000
max,5576.000000,3.630781e+07,2.337634e+06,87605.116106,99.004800,45.550000


In [6]:
nifh_simple_avg.to_csv("./csv/features/bact_data_avg2.csv")

We store the list of columns that we intend to learn from and create the model. I separated the data columns and the coordinates.

In [7]:
data_cols = ['Trichodesmium nifH Gene (x106 copies m-3)','UCYN-A nifH Gene (x106 copies m-3)','UCYN-B nifH Gene (x106 copies m-3)','Heterocyst (Richelia & Calotrhix) Biomass (mg C m-3)']
cor_cols = ['LATITUDE', 'LONGITUDE']

In [8]:
frames = []

for col in data_cols:
    #we only take a look at specific column from the dataset
    selected_data = nifh_df[cor_cols+[col]]

    #we only want to average the not na values in order not to produce nulls
    notna_mask = selected_data[col].notna()
    selected_data_notna = selected_data[notna_mask]
    print("{0}: not null values: {1}, shape: {2}".format(col, notna_mask.sum(), selected_data_notna.shape))#we can verify that we have only not null values

    #now we can average based on coordinates
    selected_data_avg = selected_data_notna.groupby(by=['LATITUDE', 'LONGITUDE']).mean()
    print("{0}: shape after averaging: {1}\n".format(col, selected_data_avg.shape))

    #we add the modified dataframe to the list
    frames.append(selected_data_avg)


Trichodesmium nifH Gene (x106 copies m-3): not null values: 2127, shape: (2127, 3)
Trichodesmium nifH Gene (x106 copies m-3): shape after averaging: (811, 1)

UCYN-A nifH Gene (x106 copies m-3): not null values: 2258, shape: (2258, 3)
UCYN-A nifH Gene (x106 copies m-3): shape after averaging: (844, 1)

UCYN-B nifH Gene (x106 copies m-3): not null values: 1760, shape: (1760, 3)
UCYN-B nifH Gene (x106 copies m-3): shape after averaging: (498, 1)

Heterocyst (Richelia & Calotrhix) Biomass (mg C m-3): not null values: 299, shape: (299, 3)
Heterocyst (Richelia & Calotrhix) Biomass (mg C m-3): shape after averaging: (117, 1)



In [9]:
#concatenate based on index(coordinates)
nifh_avg = pd.concat(frames, axis=1)

nifh_avg.head()

Trichodesmium nifH Gene (x106 copies m-3)  \
LATITUDE LONGITUDE                                              
-76.0    34.0                                        0.329000   
         35.0                                        0.057000   
-75.0    35.0                                        0.103500   
-74.0    35.0                                        1.635286   
         36.0                                        7.797333   

                    UCYN-A nifH Gene (x106 copies m-3)  \
LATITUDE LONGITUDE                                       
-76.0    34.0                                      NaN   
         35.0                                      NaN   
-75.0    35.0                                      NaN   
-74.0    35.0                                      NaN   
         36.0                                      NaN   

                    UCYN-B nifH Gene (x106 copies m-3)  \
LATITUDE LONGITUDE                                       
-76.0    34.0                                      NaN   
         35.0                                      NaN   
-75.0    35.0                                      NaN   
-74.0    35.0                                      NaN   
         36.0                                      NaN   

                    Heterocyst (Richelia & Calotrhix) Biomass (mg C m-3)  
LATITUDE LONGITUDE                                                        
-76.0    34.0                                                     NaN     
         35.0                                                     NaN     
-75.0    35.0                                                     NaN     
-74.0    35.0                                                     NaN     
         36.0                                                     NaN

In [10]:
nifh_avg.info()

<class 'pandas.core.frame.DataFrame'>
MultiIndex: 1025 entries, (-76.0, 34.0) to (57.0, 20.0)
Data columns (total 4 columns):
 #   Column                                                Non-Null Count  Dtype  
---  ------                                                --------------  -----  
 0   Trichodesmium nifH Gene (x106 copies m-3)             811 non-null    float64
 1   UCYN-A nifH Gene (x106 copies m-3)                    844 non-null    float64
 2   UCYN-B nifH Gene (x106 copies m-3)                    498 non-null    float64
 3   Heterocyst (Richelia & Calotrhix) Biomass (mg C m-3)  117 non-null    float64
dtypes: float64(4)
memory usage: 50.4 KB


In [11]:
nifh_avg.describe()

,Trichodesmium nifH Gene (x106 copies m-3),UCYN-A nifH Gene (x106 copies m-3),UCYN-B nifH Gene (x106 copies m-3),Heterocyst (Richelia & Calotrhix) Biomass (mg C m-3)
count,8.110000e+02,8.440000e+02,498.000000,117.000000
mean,6.337915e+04,3.619769e+03,266.111383,4.471009
std,1.373748e+06,8.071648e+04,3946.598824,16.774477
min,0.000000e+00,0.000000e+00,0.000000,0.000000
25%,2.306250e+00,1.937500e-01,0.000000,0.000800
50%,3.981100e+01,5.130000e+00,0.444250,0.007450
75%,2.537567e+02,8.080375e+01,8.817875,0.057300
max,3.630781e+07,2.337634e+06,87605.116106,99.004800


As we can see the results are exactly the same so a simpler one line method can be used to produce the same result